In [ ]:
سلول ۱ — مسیرها و تنظیمات اصلی

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import os
import time
import gc

PROJECT_ROOT = Path(".")

PAIR_DIR = PROJECT_ROOT / "Data_proc" / "pairs"
INTERIM_UNIPROT_DIR = PROJECT_ROOT / "Data_interim" / "uniprot"
DATA_FEAT_DIR = PROJECT_ROOT / "Data_feat"
QC_DIR = PROJECT_ROOT / "Data_proc" / "qc_reports"

FASTA_PATH = INTERIM_UNIPROT_DIR / "required_sequences_model_ready.fasta"

PAIRS_MODEL_READY_PATH = PAIR_DIR / "pairs_all_model_ready.csv"
REQ_ACCESSIONS_MODEL_PATH = PAIR_DIR / "required_accessions_model_ready.csv"

print("FASTA exists:", FASTA_PATH.exists(), FASTA_PATH)
print("Pairs model-ready exists:", PAIRS_MODEL_READY_PATH.exists())
print("Required accessions exists:", REQ_ACCESSIONS_MODEL_PATH.exists())

for d in [
    DATA_FEAT_DIR / "per_sequence",
    DATA_FEAT_DIR / "per_residue",
    QC_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

pairs_model_ready = pd.read_csv(PAIRS_MODEL_READY_PATH, dtype=str, low_memory=False)
required_accessions = pd.read_csv(REQ_ACCESSIONS_MODEL_PATH, dtype=str, low_memory=False)

print("pairs_model_ready:", pairs_model_ready.shape)
print("required_accessions:", required_accessions.shape)

display(pairs_model_ready.head())
display(required_accessions.head())

سلول ۲ — خواندن FASTA نهایی

In [ ]:
VALID_AA = set("ACDEFGHIKLMNPQRSTVWYBXZUO")

def read_multifasta(path: Path):
    records = {}
    current_id = None
    current_seq = []
    
    with open(path, "r", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            if line.startswith(">"):
                if current_id is not None:
                    records[current_id] = "".join(current_seq).upper()
                
                current_id = line[1:].split()[0].strip()
                current_seq = []
            else:
                current_seq.append(line)
        
        if current_id is not None:
            records[current_id] = "".join(current_seq).upper()
    
    return records


seqs = read_multifasta(FASTA_PATH)

print("Loaded sequences:", len(seqs))

bad_seq_rows = []

for acc, seq in seqs.items():
    invalid = sorted(set(seq) - VALID_AA)
    if len(seq) == 0 or invalid:
        bad_seq_rows.append({
            "accession": acc,
            "length": len(seq),
            "invalid_chars": "".join(invalid),
        })

bad_seq_df = pd.DataFrame(bad_seq_rows)

print("Bad sequences:", bad_seq_df.shape)
display(bad_seq_df.head())

lengths = pd.Series({acc: len(seq) for acc, seq in seqs.items()}, name="length")
display(lengths.describe())

assert len(seqs) == 2106, f"Expected 2106 sequences, got {len(seqs)}"
assert len(bad_seq_df) == 0, "There are bad sequences."

سلول ۳ — مدل‌هایی که قرار است اجرا شوند

In [ ]:
MODELS_DIR = PROJECT_ROOT / "Models"

MODEL_REGISTRY = {
    "esm2_t6_8M_UR50D": {
        "family": "hf_esm",
        "local_path": MODELS_DIR / "esm2_t6_8M_UR50D",
    },
    "esm2_t12_35M_UR50D": {
        "family": "hf_esm",
        "local_path": MODELS_DIR / "esm2_t12_35M_UR50D",
    },
    "esm2_t30_150M_UR50D": {
        "family": "hf_esm",
        "local_path": MODELS_DIR / "esm2_t30_150M_UR50D",
    },
    "esm2_t33_650M_UR50D": {
        "family": "hf_esm",
        "local_path": MODELS_DIR / "esm2_t33_650M_UR50D",
    },
    "esm2_t36_3B_UR50D": {
        "family": "hf_esm",
        "local_path": MODELS_DIR / "esm2_t36_3B_UR50D",
    },
    "prot_bert": {
        "family": "protbert",
        "local_path": MODELS_DIR / "prot_bert",
    },
    "prot_bert_bfd": {
        "family": "protbert",
        "local_path": MODELS_DIR / "prot_bert_bfd",
    },
}

MODELS_TO_RUN = [
    "esm2_t6_8M_UR50D",
    "esm2_t12_35M_UR50D",
    "esm2_t30_150M_UR50D",
    "esm2_t33_650M_UR50D",
    "esm2_t36_3B_UR50D",
    "prot_bert",
    "prot_bert_bfd",
]

for m in MODELS_TO_RUN:
    p = MODEL_REGISTRY[m]["local_path"]
    print(m, "exists:", p.exists(), p)

سلول ۴ — ابزار ذخیره خروجی و resume

In [ ]:
def prepare_model_dirs(model_name):
    seq_dir = DATA_FEAT_DIR / "per_sequence" / model_name / "proteins"
    res_dir = DATA_FEAT_DIR / "per_residue" / model_name / "proteins"
    
    seq_dir.mkdir(parents=True, exist_ok=True)
    res_dir.mkdir(parents=True, exist_ok=True)
    
    return seq_dir, res_dir


def embedding_paths(model_name, acc):
    seq_dir = DATA_FEAT_DIR / "per_sequence" / model_name / "proteins"
    res_dir = DATA_FEAT_DIR / "per_residue" / model_name / "proteins"
    
    return seq_dir / f"{acc}.npy", res_dir / f"{acc}.npy"


def already_done(model_name, acc):
    seq_path, res_path = embedding_paths(model_name, acc)
    return seq_path.exists() and res_path.exists()


def save_embedding(model_name, acc, per_sequence_vec, per_residue_mat):
    seq_path, res_path = embedding_paths(model_name, acc)
    
    np.save(seq_path, per_sequence_vec.astype(np.float32))
    np.save(res_path, per_residue_mat.astype(np.float32))


def build_embedding_index(model_name):
    seq_dir = DATA_FEAT_DIR / "per_sequence" / model_name / "proteins"
    res_dir = DATA_FEAT_DIR / "per_residue" / model_name / "proteins"
    
    rows = []
    
    for acc, seq in seqs.items():
        seq_path = seq_dir / f"{acc}.npy"
        res_path = res_dir / f"{acc}.npy"
        
        row = {
            "accession": acc,
            "sequence_length": len(seq),
            "per_sequence_path": str(seq_path),
            "per_residue_path": str(res_path),
            "has_per_sequence": seq_path.exists(),
            "has_per_residue": res_path.exists(),
            "per_sequence_dim": np.nan,
            "per_residue_shape": "",
            "ok": False,
            "error": "",
        }
        
        try:
            if seq_path.exists():
                x = np.load(seq_path, mmap_mode="r")
                row["per_sequence_dim"] = int(x.shape[0])
            
            if res_path.exists():
                r = np.load(res_path, mmap_mode="r")
                row["per_residue_shape"] = "x".join(map(str, r.shape))
                
                if len(r.shape) == 2 and r.shape[0] == len(seq):
                    row["ok"] = True
                else:
                    row["error"] = f"bad_residue_shape_expected_L={len(seq)}_got={r.shape}"
            else:
                row["error"] = "missing_per_residue"
        
        except Exception as e:
            row["error"] = str(e)
        
        rows.append(row)
    
    idx = pd.DataFrame(rows)
    
    idx_seq_path = DATA_FEAT_DIR / "per_sequence" / model_name / "proteins.index.csv"
    idx_res_path = DATA_FEAT_DIR / "per_residue" / model_name / "proteins.index.csv"
    
    idx.to_csv(idx_seq_path, index=False)
    idx.to_csv(idx_res_path, index=False)
    
    return idx

In [ ]:
import os

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

print("Offline mode enabled.")

سلول ۵ — ساخت embedding برای مدل‌های ESM-2

In [ ]:
!pip install fair-esm

In [ ]:
!pip install torch

In [ ]:
import torch
import re
import gc
from transformers import AutoTokenizer, AutoModel

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("Device:", device)


def load_hf_model_offline(local_path: Path):
    local_path = str(local_path)
    
    tokenizer = AutoTokenizer.from_pretrained(
        local_path,
        local_files_only=True,
        do_lower_case=False,
    )
    
    model = AutoModel.from_pretrained(
        local_path,
        local_files_only=True,
    )
    
    model.eval()
    model = model.to(device)
    
    return tokenizer, model

In [ ]:
@torch.no_grad()
def embed_one_hf_esm(tokenizer, model, acc, seq):
    encoded = tokenizer(
        seq,
        return_tensors="pt",
        add_special_tokens=True,
        padding=False,
        truncation=False,
    )
    
    encoded = {k: v.to(device) for k, v in encoded.items()}
    
    out = model(**encoded)
    hidden = out.last_hidden_state[0]
    
    L = len(seq)
    
    # ESM HF tokenizer: usually <cls> residues <eos>
    residue_hidden = hidden[1:L+1, :].detach().float().cpu().numpy()
    
    if residue_hidden.shape[0] != L:
        raise ValueError(
            f"ESM residue length mismatch for {acc}: expected {L}, got {residue_hidden.shape[0]}"
        )
    
    seq_vec = residue_hidden.mean(axis=0)
    
    return seq_vec, residue_hidden

سلول ۶ — اجرای یک مدل ESM

In [ ]:
def run_hf_esm_embeddings(model_name):
    spec = MODEL_REGISTRY[model_name]
    assert spec["family"] == "hf_esm"
    
    local_path = spec["local_path"]
    
    print("="*100)
    print("Running offline HF-ESM model:", model_name)
    print("Local path:", local_path)
    
    prepare_model_dirs(model_name)
    
    tokenizer, model = load_hf_model_offline(local_path)
    
    accessions = list(seqs.keys())
    log_rows = []
    
    for i, acc in enumerate(accessions):
        seq = seqs[acc]
        
        if already_done(model_name, acc):
            if i % 100 == 0:
                print(f"{model_name}: {i}/{len(accessions)} already done")
            continue
        
        try:
            if i % 20 == 0:
                print(f"{model_name}: {i}/{len(accessions)} {acc} L={len(seq)}")
            
            seq_vec, res_mat = embed_one_hf_esm(
                tokenizer=tokenizer,
                model=model,
                acc=acc,
                seq=seq,
            )
            
            save_embedding(model_name, acc, seq_vec, res_mat)
            
            log_rows.append({
                "model": model_name,
                "accession": acc,
                "status": "ok",
                "length": len(seq),
                "per_sequence_dim": seq_vec.shape[0],
                "per_residue_shape": str(res_mat.shape),
                "error": "",
            })
        
        except RuntimeError as e:
            err = str(e)
            print("RuntimeError:", acc, err[:300])
            
            log_rows.append({
                "model": model_name,
                "accession": acc,
                "status": "runtime_error",
                "length": len(seq),
                "per_sequence_dim": "",
                "per_residue_shape": "",
                "error": err[:1000],
            })
            
            if device.type == "cuda":
                torch.cuda.empty_cache()
            elif device.type == "mps":
                torch.mps.empty_cache()
        
        except Exception as e:
            err = str(e)
            print("ERROR:", acc, err[:300])
            
            log_rows.append({
                "model": model_name,
                "accession": acc,
                "status": "error",
                "length": len(seq),
                "per_sequence_dim": "",
                "per_residue_shape": "",
                "error": err[:1000],
            })
    
    log_df = pd.DataFrame(log_rows)
    log_path = QC_DIR / f"embedding_log_{model_name}.csv"
    
    if log_path.exists():
        old = pd.read_csv(log_path, dtype=str, low_memory=False)
        log_df = pd.concat([old, log_df], ignore_index=True)
    
    log_df.to_csv(log_path, index=False)
    
    idx = build_embedding_index(model_name)
    
    print("Index QC:")
    display(idx["ok"].value_counts(dropna=False))
    display(idx.head())
    
    del model
    del tokenizer
    gc.collect()
    
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()
    
    return idx

In [ ]:
idx_esm_t6 = run_hf_esm_embeddings("esm2_t6_8M_UR50D")

In [ ]:
idx_esm_t12 = run_hf_esm_embeddings("esm2_t12_35M_UR50D")

In [ ]:
idx_esm_t30 = run_hf_esm_embeddings("esm2_t30_150M_UR50D")

In [ ]:
idx_esm_t33 = run_hf_esm_embeddings("esm2_t33_650M_UR50D")

In [ ]:
idx_esm_t36 = run_hf_esm_embeddings("esm2_t36_3B_UR50D")

سلول ۷ اصلاح‌شده — ProtBERT آفلاین

In [ ]:
def load_protbert_model_offline(local_path: Path):
    local_path = str(local_path)
    
    tokenizer = AutoTokenizer.from_pretrained(
        local_path,
        local_files_only=True,
        do_lower_case=False,
    )
    
    model = AutoModel.from_pretrained(
        local_path,
        local_files_only=True,
    )
    
    model.eval()
    model = model.to(device)
    
    return tokenizer, model


def prepare_protbert_sequence(seq):
    seq = seq.upper()
    seq = re.sub(r"[UZOB]", "X", seq)
    return " ".join(list(seq))


@torch.no_grad()
def embed_one_protbert(tokenizer, model, acc, seq):
    spaced = prepare_protbert_sequence(seq)
    
    encoded = tokenizer(
        spaced,
        return_tensors="pt",
        add_special_tokens=True,
        padding=False,
        truncation=False,
    )
    
    encoded = {k: v.to(device) for k, v in encoded.items()}
    
    out = model(**encoded)
    hidden = out.last_hidden_state[0]
    
    L = len(seq)
    residue_hidden = hidden[1:L+1, :].detach().float().cpu().numpy()
    
    if residue_hidden.shape[0] != L:
        raise ValueError(
            f"ProtBERT residue length mismatch for {acc}: expected {L}, got {residue_hidden.shape[0]}"
        )
    
    seq_vec = residue_hidden.mean(axis=0)
    return seq_vec, residue_hidden

سلول ۸ اصلاح‌شده — اجرای ProtBERT آفلاین

In [ ]:
def run_protbert_embeddings(model_name):
    spec = MODEL_REGISTRY[model_name]
    assert spec["family"] == "protbert"
    
    local_path = spec["local_path"]
    
    print("="*100)
    print("Running offline ProtBERT model:", model_name)
    print("Local path:", local_path)
    
    prepare_model_dirs(model_name)
    
    tokenizer, model = load_protbert_model_offline(local_path)
    
    accessions = list(seqs.keys())
    log_rows = []
    
    for i, acc in enumerate(accessions):
        seq = seqs[acc]
        
        if already_done(model_name, acc):
            if i % 100 == 0:
                print(f"{model_name}: {i}/{len(accessions)} already done")
            continue
        
        try:
            if i % 20 == 0:
                print(f"{model_name}: {i}/{len(accessions)} {acc} L={len(seq)}")
            
            seq_vec, res_mat = embed_one_protbert(
                tokenizer=tokenizer,
                model=model,
                acc=acc,
                seq=seq,
            )
            
            save_embedding(model_name, acc, seq_vec, res_mat)
            
            log_rows.append({
                "model": model_name,
                "accession": acc,
                "status": "ok",
                "length": len(seq),
                "per_sequence_dim": seq_vec.shape[0],
                "per_residue_shape": str(res_mat.shape),
                "error": "",
            })
        
        except RuntimeError as e:
            err = str(e)
            print("RuntimeError:", acc, err[:300])
            
            log_rows.append({
                "model": model_name,
                "accession": acc,
                "status": "runtime_error",
                "length": len(seq),
                "per_sequence_dim": "",
                "per_residue_shape": "",
                "error": err[:1000],
            })
            
            if device.type == "cuda":
                torch.cuda.empty_cache()
            elif device.type == "mps":
                torch.mps.empty_cache()
        
        except Exception as e:
            err = str(e)
            print("ERROR:", acc, err[:300])
            
            log_rows.append({
                "model": model_name,
                "accession": acc,
                "status": "error",
                "length": len(seq),
                "per_sequence_dim": "",
                "per_residue_shape": "",
                "error": err[:1000],
            })
    
    log_df = pd.DataFrame(log_rows)
    log_path = QC_DIR / f"embedding_log_{model_name}.csv"
    
    if log_path.exists():
        old = pd.read_csv(log_path, dtype=str, low_memory=False)
        log_df = pd.concat([old, log_df], ignore_index=True)
    
    log_df.to_csv(log_path, index=False)
    
    idx = build_embedding_index(model_name)
    
    print("Index QC:")
    display(idx["ok"].value_counts(dropna=False))
    display(idx.head())
    
    del model
    del tokenizer
    gc.collect()
    
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()
    
    return idx

In [ ]:
idx_protbert = run_protbert_embeddings("prot_bert")

In [ ]:
idx_protbert_bfd = run_protbert_embeddings("prot_bert_bfd")

سلول ۹ — QC کلی embedding coverage برای همه مدل‌ها

In [ ]:
def summarize_embedding_coverage(model_names):
    rows = []
    
    for model_name in model_names:
        idx_path = DATA_FEAT_DIR / "per_sequence" / model_name / "proteins.index.csv"
        
        if not idx_path.exists():
            rows.append({
                "model": model_name,
                "index_exists": False,
                "n_required": len(seqs),
                "n_ok": 0,
                "n_missing": len(seqs),
                "coverage": 0.0,
                "dims": "",
                "errors": "missing_index",
            })
            continue
        
        idx = pd.read_csv(idx_path, dtype=str, low_memory=False)
        
        idx["ok_bool"] = idx["ok"].astype(str).str.lower().eq("true")
        
        dims = sorted(idx.loc[idx["ok_bool"], "per_sequence_dim"].dropna().astype(str).unique())
        
        errors = (
            idx.loc[~idx["ok_bool"], "error"]
            .fillna("")
            .replace("", "unknown")
            .value_counts()
            .head(5)
            .to_dict()
        )
        
        rows.append({
            "model": model_name,
            "index_exists": True,
            "n_required": len(idx),
            "n_ok": int(idx["ok_bool"].sum()),
            "n_missing": int((~idx["ok_bool"]).sum()),
            "coverage": float(idx["ok_bool"].mean()),
            "dims": ";".join(dims),
            "errors": str(errors),
        })
    
    return pd.DataFrame(rows)


embedding_coverage_summary = summarize_embedding_coverage(MODELS_TO_RUN)

display(embedding_coverage_summary)

embedding_coverage_summary.to_csv(
    QC_DIR / "embedding_coverage_summary_all_models.csv",
    index=False
)